In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
from google.colab import files
uploaded = files.upload()

Saving Updated_Merged_Data.csv to Updated_Merged_Data.csv


In [3]:
df = pd.read_csv("/content/Updated_Merged_Data.csv")
df.head()

,timestamp,satellite_prn,satellite_elevation_deg,satellite_azimuth_deg,temperature_C,humidity_%,rain_intensity_mm_hr,rain_sensor_value,snr_measured_dbhz,label
0,2026-02-01 08:10,8.23,44.94,313.18,34.99,58.64,0.00,0.00,51,0
1,2026-01-25 02:12,2.94,22.20,200.38,33.96,55.17,0.57,0.00,47,0
2,2026-03-01 11:18,11.30,26.44,162.51,32.92,50.66,0.00,0.00,42,0
3,2026-02-21 07:00,26.70,77.68,58.32,31.32,40.02,1.36,0.00,44,0
4,2026-03-04 06:52,21.89,82.50,54.38,28.65,89.37,2.79,49.88,39,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   timestamp                30000 non-null  object 
 1   satellite_prn            30000 non-null  float64
 2   satellite_elevation_deg  30000 non-null  float64
 3   satellite_azimuth_deg    30000 non-null  float64
 4   temperature_C            30000 non-null  float64
 5   humidity_%               30000 non-null  float64
 6   rain_intensity_mm_hr     30000 non-null  float64
 7   rain_sensor_value        30000 non-null  float64
 8   snr_measured_dbhz        30000 non-null  int64  
 9   label                    30000 non-null  int64  
dtypes: float64(7), int64(2), object(1)
memory usage: 2.3+ MB


In [8]:
print(df.isnull().sum())

timestamp                  0
satellite_prn              0
satellite_elevation_deg    0
satellite_azimuth_deg      0
temperature_C              0
humidity_%                 0
rain_intensity_mm_hr       0
rain_sensor_value          0
snr_measured_dbhz          0
label                      0
dtype: int64


In [10]:
(df == 0).sum()

,0
timestamp,0
satellite_prn,925
satellite_elevation_deg,883
satellite_azimuth_deg,951
temperature_C,904
humidity_%,930
rain_intensity_mm_hr,6087
rain_sensor_value,6347
snr_measured_dbhz,894
label,11982


In [11]:
cols = [
    "temperature_C",
    "humidity_%"
]

for col in cols:
    df[col] = df[col].replace(0, np.nan)
    df[col] = df[col].fillna(df[col].mean())

In [13]:
(df == 0).sum()

,0
timestamp,0
satellite_prn,925
satellite_elevation_deg,883
satellite_azimuth_deg,951
temperature_C,0
humidity_%,0
rain_intensity_mm_hr,6087
rain_sensor_value,6347
snr_measured_dbhz,894
label,11982


In [14]:
df = df[
    (df["satellite_elevation_deg"] >= 10) &
    (df["satellite_elevation_deg"] <= 70) &
    (df["satellite_azimuth_deg"] >= 0) &
    (df["satellite_azimuth_deg"] <= 360) &
    (df["satellite_prn"] >= 1) &
    (df["satellite_prn"] <= 32)
]

In [15]:
print("Filtered dataset size:", df.shape)

Filtered dataset size: (20328, 10)


In [16]:
features = [
    "temperature_C",
    "humidity_%",
    "snr_measured_dbhz",
    "rain_sensor_value"
]

X = df[features]
y = df["label"]

In [18]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)

In [20]:
model = LogisticRegression(
    multi_class="multinomial",
    solver="lbfgs",
    max_iter=1000
)

model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


LogisticRegression(max_iter=1000, multi_class='multinomial')

In [21]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Model Accuracy:", accuracy)

Model Accuracy: 0.9827840629611412


In [22]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.99      0.99      1627
           1       0.96      0.99      0.97      1212
           2       0.99      0.97      0.98      1227

    accuracy                           0.98      4066
   macro avg       0.98      0.98      0.98      4066
weighted avg       0.98      0.98      0.98      4066



In [23]:
print(confusion_matrix(y_test, y_pred))

[[1612   15    0]
 [   5 1194   13]
 [   0   37 1190]]


In [24]:
print("Model Coefficients:")
print(model.coef_)

print("Intercept:")
print(model.intercept_)

Model Coefficients:
[[ 11.03188626 -16.75532045   4.14548722  -8.36817677]
 [  0.43147848   3.77895846   0.76796779  -3.00623068]
 [-11.46336474  12.97636199  -4.91345502  11.37440745]]
Intercept:
[ 1.32955736  0.12708801 -1.45664538]


In [25]:
np.save("weights.npy", model.coef_)
np.save("bias.npy", model.intercept_)

In [27]:
print("Feature Min:", scaler.data_min_)
print("Feature Max:", scaler.data_max_)

Feature Min: [20.36 29.51  0.    0.  ]
Feature Max: [ 37.81 107.72  56.   831.15]
